# Module 2 — Differential Gene Expression

**Goal:** Identify genes whose expression level differs significantly between each PAM50 subtype and all other subtypes combined (one-vs-rest).

**Method:**

- **Input data:** UCSC Xena stores STAR counts as log₂(count + 1) — the file is already normalized, so no further transform is applied.
- **Test:** Welch's t-test (does not assume equal variance — appropriate because group sizes differ).
- **Multiple testing:** 5 subtypes × 5 000 genes = 25 000 tests. Benjamini-Hochberg FDR correction applied within each subtype's 5 000 tests.
- **Significance threshold:** FDR < 0.05 **and** |log₂FC| ≥ 1.0 (at least 2× fold change).

A gene with log₂FC > 0 is **upregulated** in the subtype; log₂FC < 0 means **downregulated**.

In [ ]:
import sys
sys.path.insert(0, "..")

import logging
import pandas as pd
import matplotlib.pyplot as plt

from src.constants import InterimFile, KNOWN_MARKERS
from src.paths import interim_file
from src.features.differential_expression import run_one_vs_rest, top_de_genes
from src.visualization.volcano import plot_all_volcanos, plot_volcano

logging.basicConfig(level=logging.INFO, format="%(message)s")

## Load Data


In [ ]:
X = pd.read_csv(interim_file(InterimFile.X), index_col=0)
labels = pd.read_csv(interim_file(InterimFile.LABELS), index_col=0).squeeze()

print(f"X:      {X.shape[0]:>5} samples x {X.shape[1]:>5} genes")
print(f"labels: {len(labels):>5} samples")
print()
print(labels.value_counts())

## Run Differential Expression


In [ ]:
FDR_THRESHOLD = 0.05
LOG2FC_THRESHOLD = 1.0  # >= 2x fold change

results = run_one_vs_rest(
    X=X,
    labels=labels,
    fdr_threshold=FDR_THRESHOLD,
    log2fc_threshold=LOG2FC_THRESHOLD,
)
results.head(3)

## Significant Genes per Subtype

How many gene-subtype pairs pass the threshold, broken down by direction?


In [ ]:
sig = results[results["significant"]].copy()
sig["direction"] = sig["log2fc"].apply(lambda x: "up" if x > 0 else "down")

summary = (
    sig.groupby(["subtype", "direction"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["up", "down"])
)
summary["total"] = summary.sum(axis=1)
summary.sort_values("total", ascending=False)

## Top DE Genes per Subtype

Top 10 upregulated genes per subtype (ranked by adjusted p-value).

> **Note:** Gene IDs are Ensembl IDs with version suffix (e.g. `ENSG00000141736.14`). A future step would map these to HGNC symbols via biomaRt or MyGene.info.


In [ ]:
top_up = top_de_genes(results, n=10, direction="up")

display_cols = ["subtype", "gene", "log2fc", "padj", "mean_subtype", "mean_rest"]
(
    top_up[display_cols]
    .style
    .format({"log2fc": "{:.2f}", "padj": "{:.2e}",
             "mean_subtype": "{:.2f}", "mean_rest": "{:.2f}"})
    .background_gradient(subset="log2fc", cmap="Reds")
)

## Volcano Plots

Each point is one gene. x-axis = log₂FC (positive = higher in this subtype). y-axis = -log₁₀(FDR-adjusted p-value). Red = significantly up, blue = significantly down, grey = not significant. Dashed lines mark the thresholds.


In [ ]:
fig = plot_all_volcanos(
    results,
    fdr_threshold=FDR_THRESHOLD,
    log2fc_threshold=LOG2FC_THRESHOLD,
    n_label=8,
    ncols=3,
)
plt.show()

## Known Marker Validation

A credible DGE result should recover established PAM50 biomarkers. We check whether canonical markers from the literature appear in our significant genes.

Genes are stored as versioned Ensembl IDs (e.g. `ENSG00000141736.14`), so we match on the base ID.


In [ ]:
results["gene_base"] = results["gene"].str.split(".").str[0]

rows = []
for subtype, marker_ids in KNOWN_MARKERS.items():
    for ensembl_id in marker_ids:
        hit = results[
            (results["subtype"] == subtype) & (results["gene_base"] == ensembl_id)
        ]
        if hit.empty:
            rows.append({"subtype": subtype, "ensembl_id": ensembl_id,
                         "in_top5000": False, "significant": False,
                         "log2fc": None, "padj": None})
        else:
            r = hit.iloc[0]
            rows.append({"subtype": subtype, "ensembl_id": ensembl_id,
                         "in_top5000": True, "significant": bool(r["significant"]),
                         "log2fc": round(float(r["log2fc"]), 2),
                         "padj": f"{r['padj']:.2e}"})

marker_df = pd.DataFrame(rows)
print(f"Markers in top-5000 variable genes: {marker_df['in_top5000'].sum()} / {len(marker_df)}")
print(f"Markers also significant:           {marker_df['significant'].sum()} / {marker_df['in_top5000'].sum()}")
marker_df

## Save Results


In [ ]:
out = interim_file(InterimFile.DE_RESULTS)
results.drop(columns="gene_base", errors="ignore").to_csv(out, index=False)
print(f"Saved {out.name}  ({len(results):,} rows)")